In [36]:
# 02_data_preprocessing.ipynb

# --- CELL 1: Setup and Imports ---
from __future__ import annotations

import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

# Set a global seed for full reproducibility
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("./data/raw")
OUTPUT_DIR = Path("./data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_FILE = DATA_DIR / "AngioCAD_Features.xlsx"
LABELS_FILE = DATA_DIR / "AngioCAD_Labels.xlsx"

print(f"Output directory: {OUTPUT_DIR}")


Output directory: data\processed


In [37]:
# --- CELL 2: Load Raw Data ---
df_features = pd.read_excel(FEATURES_FILE)
df_labels = pd.read_excel(LABELS_FILE)

print(f"Features: {df_features.shape}")
print(f"Labels  : {df_labels.shape}")


Features: (377, 60)
Labels  : (413, 18)


In [38]:

# --- CELL 3: Parse Stenosis Columns ---
# The raw stenosis values are strings like "NL", "50", "26-50". We convert
# them to a numeric midpoint.

STENOSIS_COLS = [
    "LM", "Prox LAD", "Mid LAD", "Dist LAD", "1st dig", "2nd dig",
    "Prox LCX", "Mid LCX", "Dist LCX", "OM",
    "Prox RCA", "Mid RCA", "Dist RCA", "PDA", "PLB",
]

def parse_stenosis(value):
    """Convert a raw stenosis annotation to a numeric midpoint."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        token = value.strip().upper()
        if token == "NL":
            return 0.0
        if "-" in token:
            parts = token.split("-")
            try:
                return (float(parts[0]) + float(parts[1])) / 2.0
            except (ValueError, IndexError):
                return np.nan
        try:
            return float(token)
        except ValueError:
            return np.nan
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

for col in STENOSIS_COLS:
    if col in df_labels.columns:
        df_labels[f"{col}_num"] = df_labels[col].apply(parse_stenosis)

NUM_STENOSIS = [f"{c}_num" for c in STENOSIS_COLS if f"{c}_num" in df_labels.columns]
print(f"Parsed {len(NUM_STENOSIS)} stenosis columns")

Parsed 15 stenosis columns


In [39]:
# --- CELL 4: Build Target Labels ---
# We create a binary CAD label (>= 50% stenosis) and a multi-class severity label.
df_labels["max_stenosis"] = df_labels[NUM_STENOSIS].max(axis=1, skipna=True)

# Remove patients with no interpretable annotation
missing = df_labels["max_stenosis"].isna()
print(f"Excluded (no annotation): {int(missing.sum())}")
df_labels = df_labels.loc[~missing].copy()

# Binary CAD target
df_labels["has_cad"] = (df_labels["max_stenosis"] >= 50).astype(int)

# Severity classification
def severity_from_stenosis(value: float) -> str:
    if value == 0:      return "Normal"
    if value < 30:      return "Mild"
    if value < 50:      return "Mild-Moderate"
    if value < 70:      return "Moderate"
    return "Severe"

df_labels["severity"] = df_labels["max_stenosis"].apply(severity_from_stenosis)
df_labels["num_diseased_vessels"] = df_labels[NUM_STENOSIS].ge(50).sum(axis=1)

print("\nCAD distribution:")
print(df_labels["has_cad"].value_counts())
print(f"Prevalence: {df_labels['has_cad'].mean():.4f}")

Excluded (no annotation): 0

CAD distribution:
has_cad
1    324
0     89
Name: count, dtype: int64
Prevalence: 0.7845


In [40]:
# --- CELL 5: Merge Features and Labels ---
if df_features["ID"].duplicated().any():
    raise ValueError("Duplicate IDs in features.")
if df_labels["ID"].duplicated().any():
    raise ValueError("Duplicate IDs in labels.")

df = df_features.merge(df_labels, on="ID", how="inner", validate="one_to_one")
print(f"\nMerged: {df.shape}, unique patients: {df['ID'].nunique()}")


Merged: (377, 96), unique patients: 377


In [41]:
# --- CELL 6: Feature Schema ---
NUMERIC_FEATURES = [
    "Age", "Length", "Weight", "BMI",
    "LVEF", "LVEF dysfunction",
    "TG", "Cholesterol", "HDL", "LDL", "FBS",
    "Cr", "BUN", "Hgb", "Hct",
    "leukocytes", "Neut", "Monocyte", "MXD", "Lymph",
    "PLT", "RDW",
]

CATEGORICAL_FEATURES = [
    "Sex", "dominance",
    "DM", "HTN", "Hyperlipidemia", "Current Smoker",
    "PVD", "Stroke",
    "Beta blocker", "Alpha blocker", "Diuretic",
    "ACEI", "ARB", "CCB", "ASA", "Clopidogrel", "Astatine",
]

NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c in df.columns]
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in df.columns]

print(f"Numeric     : {len(NUMERIC_FEATURES)}")
print(f"Categorical : {len(CATEGORICAL_FEATURES)}")

Numeric     : 21
Categorical : 17


In [42]:
# --- CELL 7: Stratified Splits ---
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[FEATURE_COLS].copy()
y_binary = df["has_cad"].astype(int)
y_severity = df["severity"].copy()

# Patient-level stratified split (80/10/10 approximately)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y_binary, test_size=0.20, stratify=y_binary, random_state=SEED,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15, stratify=y_trainval, random_state=SEED,
)

# Track severity labels for each split
y_severity_train = y_severity.loc[X_train.index]
y_severity_val = y_severity.loc[X_val.index]
y_severity_test = y_severity.loc[X_test.index]

print(f"Train : {len(X_train):4d}  CAD+: {int(y_train.sum()):4d}")
print(f"Val   : {len(X_val):4d}  CAD+: {int(y_val.sum()):4d}")
print(f"Test  : {len(X_test):4d}  CAD+: {int(y_test.sum()):4d}")

Train :  255  CAD+:  202
Val   :   46  CAD+:   36
Test  :   76  CAD+:   60


In [43]:
# --- CELL 8: Preprocessing Pipeline (FIT ON TRAIN ONLY) ---
# This is the crucial anti-leakage step. The preprocessor is fit on the
# training data and then used to transform validation and test data.

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATURES),
    ("cat", categorical_pipe, CATEGORICAL_FEATURES),
])

X_train_p = preprocessor.fit_transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out().tolist()

print(f"\nProcessed dimensions: {X_train_p.shape}")
print(f"Feature names count : {len(feature_names)}")





Processed dimensions: (255, 55)
Feature names count : 55


In [44]:
# --- CELL 9: Persist Numeric Table for Downstream Notebooks ---
X_all_p = preprocessor.transform(X)
clinical_numeric = pd.DataFrame(X_all_p, columns=feature_names, index=X.index)
clinical_numeric.insert(0, "ID", df["ID"].values)
clinical_numeric["has_cad"] = df["has_cad"].values
clinical_numeric["max_stenosis"] = df["max_stenosis"].values
clinical_numeric["num_diseased_vessels"] = df["num_diseased_vessels"].values

non_numeric = clinical_numeric.select_dtypes(exclude="number").columns.tolist()
assert not non_numeric, f"Non-numeric columns remain: {non_numeric}"

clinical_numeric.to_csv(OUTPUT_DIR / "clinical_features_merged.csv", index=False)
print(f"clinical_features_merged.csv -> {clinical_numeric.shape}")

# Save original table (needed for image extraction and VLM prompts)
df.to_csv(OUTPUT_DIR / "clinical_features_original.csv", index=False)

# Save auxiliary labels
pd.DataFrame({"ID": df["ID"].values, "severity": df["severity"].values}) \
    .to_csv(OUTPUT_DIR / "severity_labels.csv", index=False)
pd.DataFrame({"ID": df["ID"].values, "Diagnosis": df["Diagnosis"].values}) \
    .to_csv(OUTPUT_DIR / "diagnosis_labels.csv", index=False)

print("Auxiliary tables saved.")

clinical_features_merged.csv -> (377, 59)
Auxiliary tables saved.


In [45]:
# --- CELL 10: Save Splits, Processed Arrays, and Artifacts ---
splits = {
    "train": [int(x) for x in df.loc[X_train.index, "ID"]],
    "val":   [int(x) for x in df.loc[X_val.index, "ID"]],
    "test":  [int(x) for x in df.loc[X_test.index, "ID"]],
}
with open(OUTPUT_DIR / "splits.json", "w") as f:
    json.dump(splits, f, indent=2)

np.save(OUTPUT_DIR / "X_train_processed.npy", X_train_p)
np.save(OUTPUT_DIR / "X_val_processed.npy", X_val_p)
np.save(OUTPUT_DIR / "X_test_processed.npy", X_test_p)
np.save(OUTPUT_DIR / "y_train.npy", y_train.to_numpy())
np.save(OUTPUT_DIR / "y_val.npy", y_val.to_numpy())
np.save(OUTPUT_DIR / "y_test.npy", y_test.to_numpy())

with open(OUTPUT_DIR / "preprocessing_artifacts.pkl", "wb") as f:
    pickle.dump({
        "preprocessor": preprocessor,
        "feature_cols": FEATURE_COLS,
        "numeric_cols": NUMERIC_FEATURES,
        "categorical_cols": CATEGORICAL_FEATURES,
        "feature_names_out": feature_names,
        "seed": SEED,
    }, f)

# Compute class weights for the imbalanced training set
class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight_map = {int(c): float(w) for c, w in zip(np.unique(y_train), class_weights)}
with open(OUTPUT_DIR / "class_weights.json", "w") as f:
    json.dump(class_weight_map, f, indent=2)

print(f"Class weights: {class_weight_map}")
print("All artifacts saved.")

Class weights: {0: 2.4056603773584904, 1: 0.6311881188118812}
All artifacts saved.
